# Workout Class Classifier - Gradient Boosted Trees

This notebook trains a Gradient Boosted Trees classifier to predict workout classes from pose sequences.

The model:
- Takes sequences of shape (15, 12, 3) - 15 frames, 12 landmarks, 3 coordinates
- Flattens them to feature vectors
- Predicts workout classes

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

## Load Training Data

In [ ]:
# Load training data from .npz file
data_path = Path('../../Data/output/training_data.npz')
metadata_path = Path('../../Data/output/training_data_metadata.json')

# Load metadata
with open(metadata_path, 'r') as f:
    metadata = json.load(f)

print("Dataset Metadata:")
print(f"  Number of samples: {metadata['n_samples']}")
print(f"  Number of classes: {metadata['n_classes']}")
print(f"  Sequence length: {metadata['sequence_length']}")
print(f"  Number of landmarks: {metadata['n_landmarks']}")
print(f"  Number of coordinates: {metadata['n_coords']}")
print(f"  Feature shape: {metadata['feature_shape']}")
print(f"\nClass names: {metadata['class_names']}")
print(f"\nClass distribution:")
for cls, count in metadata['class_distribution'].items():
    print(f"  {cls}: {count}")

# Load the .npz file
data = np.load(data_path, allow_pickle=True)

# Check available keys
print(f"\nAvailable keys in .npz file: {list(data.keys())}")

# Load data - note: the key is 'y' not 'y_encoded' based on how it was saved
X = data['X']  # Shape: (n_samples, 15, 33, 3)
y_encoded = data['y']  # Encoded labels (integers) - key is 'y' not 'y_encoded'
y_raw = data['y_raw']  # Raw class names
class_names = data['class_names']  # Class names array

print(f"\nLoaded data shapes:")
print(f"  X shape: {X.shape}")
print(f"  y_encoded shape: {y_encoded.shape}")
print(f"  Number of classes: {len(class_names)}")

## Preprocess Data

Flatten the sequences from (15, 33, 3) to a 1D feature vector for the gradient boosting model.

In [ ]:
# Flatten sequences: (n_samples, 15, 33, 3) -> (n_samples, 15*33*3)
X_flat = X.reshape(X.shape[0], -1)
print(f"Flattened X shape: {X_flat.shape}")
print(f"Number of features per sample: {X_flat.shape[1]}")

# Check for NaN or infinite values
print(f"\nData quality check:")
print(f"  NaN values in X: {np.isnan(X_flat).sum()}")
print(f"  Infinite values in X: {np.isinf(X_flat).sum()}")

# Replace any NaN or infinite values with 0
X_flat = np.nan_to_num(X_flat, nan=0.0, posinf=0.0, neginf=0.0)

## Split Data into Train and Test Sets

In [ ]:
# Split data into train and test sets
# Using y_encoded for splitting (integer labels)
X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X_flat, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Ensure balanced split across classes
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Feature dimension: {X_train.shape[1]}")
print(f"Number of classes: {len(class_names)}")

## Train Gradient Boosted Trees Classifier

We'll use XGBoost which is a powerful gradient boosting framework that supports multi-class classification.

In [ ]:
# Initialize XGBoost classifier for multi-class classification
model = xgb.XGBClassifier(
    objective='multi:softprob',  # Multi-class classification with probability output
    num_class=len(class_names),    # Number of classes
    n_estimators=200,              # Number of boosting rounds
    max_depth=6,                   # Maximum tree depth
    learning_rate=0.1,             # Learning rate
    subsample=0.8,                 # Subsample ratio of training instances
    colsample_bytree=0.8,         # Subsample ratio of columns when constructing each tree
    random_state=42,
    n_jobs=-1,                     # Use all available cores
    eval_metric='mlogloss'         # Multi-class log loss
)

print("Training XGBoost classifier...")
# Train on encoded labels (integers) - XGBoost will handle multi-class internally
model.fit(
    X_train, 
    y_train_encoded,
    eval_set=[(X_train, y_train_encoded), (X_test, y_test_encoded)],
    verbose=True
)

print("\nTraining completed!")

## Evaluate Model

Make predictions and evaluate the model performance.

In [ ]:
# Make predictions
y_pred_proba = model.predict_proba(X_test)  # Shape: (n_test_samples, n_classes)
y_pred_encoded = model.predict(X_test)      # Predicted class indices

print("Predictions shape:")
print(f"  y_pred_proba shape: {y_pred_proba.shape}")
print(f"  y_pred_encoded shape: {y_pred_encoded.shape}")

# Calculate accuracy
accuracy = accuracy_score(y_test_encoded, y_pred_encoded)
print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Detailed classification report
print("\nClassification Report:")
print(classification_report(
    y_test_encoded, 
    y_pred_encoded, 
    target_names=class_names,
    digits=3
))

## Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test_encoded, y_pred_encoded)

# Plot confusion matrix
plt.figure(figsize=(16, 14))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix - Workout Class Classifier', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=12)
plt.ylabel('True Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print("\nPer-Class Accuracy:")
for i, cls_name in enumerate(class_names):
    print(f"  {cls_name}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")

## Feature Importance

Visualize which features (landmarks/coordinates) are most important for classification.

In [ ]:
# Get feature importance
feature_importance = model.feature_importances_

# Get top 30 most important features
top_n = 100
top_indices = np.argsort(feature_importance)[-top_n:][::-1]
top_importance = feature_importance[top_indices]

# Plot feature importance
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_importance)), top_importance)
plt.yticks(range(len(top_importance)), [f'Feature {i}' for i in top_indices])
plt.xlabel('Feature Importance', fontsize=12)
plt.ylabel('Feature Index', fontsize=12)
plt.title(f'Top {top_n} Most Important Features', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop {top_n} most important features:")
for idx, imp in zip(top_indices[:10], top_importance[:10]):
    # Calculate which frame, landmark, and coordinate this feature represents
    frame_idx = idx // (33 * 3)
    remainder = idx % (33 * 3)
    landmark_idx = remainder // 3
    coord_idx = remainder % 3
    coord_name = ['x', 'y', 'z'][coord_idx]
    print(f"  Feature {idx}: Importance={imp:.6f} (Frame {frame_idx}, Landmark {landmark_idx}, {coord_name})")

## Example Predictions

Show some example predictions with probabilities.

In [ ]:
# Show some example predictions
n_examples = 10
example_indices = np.random.choice(len(X_test), n_examples, replace=False)

print("Example Predictions:")
print("=" * 80)
for idx in example_indices:
    true_class_idx = y_test_encoded[idx]
    pred_class_idx = y_pred_encoded[idx]
    true_class = class_names[true_class_idx]
    pred_class = class_names[pred_class_idx]
    probabilities = y_pred_proba[idx]
    
    # Get top 3 predicted classes
    top3_indices = np.argsort(probabilities)[-3:][::-1]
    
    print(f"\nSample {idx}:")
    print(f"  True class: {true_class}")
    print(f"  Predicted class: {pred_class} {'✓' if true_class == pred_class else '✗'}")
    print(f"  Top 3 predictions:")
    for i, class_idx in enumerate(top3_indices):
        print(f"    {i+1}. {class_names[class_idx]}: {probabilities[class_idx]:.4f}")

## Save Model

Save the trained model for future use.

In [ ]:
import pickle
from pathlib import Path

# Create output directory if it doesn't exist
output_dir = Path('models')
output_dir.mkdir(exist_ok=True)

# Save the model
model_path = output_dir / 'xgboost_workout_classifier.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(model, f)

# Save class names for reference
class_names_path = output_dir / 'class_names.json'
with open(class_names_path, 'w') as f:
    json.dump(class_names.tolist(), f, indent=2)

# Save metadata about the model
model_metadata = {
    'model_type': 'XGBoost',
    'n_classes': len(class_names),
    'n_features': X_flat.shape[1],
    'feature_shape': metadata['feature_shape'],
    'class_names': class_names.tolist(),
    'test_accuracy': float(accuracy),
    'model_path': str(model_path),
    'class_names_path': str(class_names_path)
}

metadata_path = output_dir / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(model_metadata, f, indent=2)

print(f"Model saved to: {model_path}")
print(f"Class names saved to: {class_names_path}")
print(f"Model metadata saved to: {metadata_path}")

## Prediction Function

Helper function to make predictions on new sequences.

In [ ]:
def predict_workout_class(sequence, model, class_names, return_proba=False):
    """
    Predict workout class for a sequence.
    
    Args:
        sequence: numpy array of shape (15, 33, 3) or (1, 15, 33, 3)
        model: Trained XGBoost model
        class_names: Array of class names
        return_proba: If True, return probabilities for all classes, else return class name
    
    Returns:
        If return_proba=True: probability array of shape (n_classes,)
        If return_proba=False: predicted class name (string)
    """
    # Ensure sequence is the right shape
    if sequence.ndim == 3:
        sequence = sequence.reshape(1, -1)  # Flatten to (1, 15*33*3)
    elif sequence.ndim == 4:
        sequence = sequence.reshape(sequence.shape[0], -1)  # Flatten to (n, 15*33*3)
    else:
        sequence = sequence.reshape(1, -1)  # Flatten to (1, n_features)
    
    # Handle NaN and infinite values
    sequence = np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)
    
    # Get probabilities
    probabilities = model.predict_proba(sequence)
    
    if return_proba:
        # Return probabilities
        return probabilities[0] if probabilities.shape[0] == 1 else probabilities
    else:
        # Return class name
        pred_class_idx = np.argmax(probabilities, axis=1)
        if len(pred_class_idx) == 1:
            return class_names[pred_class_idx[0]]
        else:
            return [class_names[idx] for idx in pred_class_idx]

# Test the prediction function
print("Testing prediction function:")
test_sequence = X_test[0:1].reshape(1, 15, 12, 3)  # Reshape back to original format
pred_class = predict_workout_class(test_sequence, model, class_names, return_proba=False)
pred_proba = predict_workout_class(test_sequence, model, class_names, return_proba=True)

print(f"  Input shape: {test_sequence.shape}")
print(f"  Predicted class: {pred_class}")
print(f"  Probabilities shape: {pred_proba.shape}")
print(f"  Top 3 class probabilities:")
top3_indices = np.argsort(pred_proba)[-3:][::-1]
for i, idx in enumerate(top3_indices):
    print(f"    {i+1}. {class_names[idx]}: {pred_proba[idx]:.4f}")
print(f"  True class: {class_names[y_test_encoded[0]]}")